## Run the cell below to copy a new overlay form /home/xilinx 

In [ ]:
! mv /home/xilinx/main.* .

## The code below starts a waveform

In [ ]:
from pynq import Overlay, MMIO
from xrfdc import RFdc
from pynq.overlays.base import BaseOverlay
import time

class ESBL(BaseOverlay):
    def __init__(self):
        Overlay.__init__(self,'main.bit')
        if self.is_loaded():
            self.i2c_initialized = False
            self.display_port_initialized = False
            self.radio = self.usp_rf_data_converter_0
            self.reg = MMIO(overlay.ip_dict['ESBL_control_reg']['phys_addr'],
                            overlay.ip_dict['ESBL_control_reg']['addr_range']).array
            self.init_rf_clks()
            
            esbl.freq = 0.001
            self.radio.dac_tiles[0].blocks[0].MixerSettings['EventSource'] = 0
            self.radio.dac_tiles[0].blocks[0].MixerSettings['FineMixerScale'] = 1
            esbl.freq = 0.0
            
            self.setup()
    
    @property 
    def freq(self): # MHz
        return self.radio.dac_tiles[0].blocks[0].MixerSettings['Freq']
    
    
    @freq.setter
    def freq(self,freq): # MHz
        self.radio.dac_tiles[0].blocks[0].MixerSettings['Freq'] = freq
        
    def setup(self):
        #set the phase difference to 90 deg
        self.reg[2] = 0x00100000

        #set i_ampl (16 bits)
        self.reg[3] = 0x0000FFFF

        #set q_ampl (16 bits)
        self.reg[4] = 0x0000FFFF

    def set_PDH_phase(self,phase):
        #in machine units (20 bits)
        self.reg[5]= phase

    def set_mod_frequency(self,frequency, sampling_rate = 153_600_000):    
        frequency_code = int(frequency * 0xFFFF_FFFF / sampling_rate / 8) # divided by 8 because the PDH generator runs 8 times faster
        self.reg[0] = frequency_code

    def change_mod_depth(self,depth=42):
        # 8 bits
        self.reg[1] = depth

    
            

esbl = ESBL()

esbl.freq = 350


change_mod_depth(42)
set_mod_frequency(30_000_000)





In [ ]:
for n in range(100):
    esbl.freq = n * 20
    time.sleep(0.1)


